In [3]:
import glob, os
import numpy as np
import pandas as pd

RANDOM_SEED = 42
ID_COL = "user_id"

csv_files = glob.glob(os.path.join('./data/', '*c000.csv'))
df_all = pd.concat((pd.read_csv(f, sep=";") for f in csv_files), ignore_index=True)
forget_df = pd.read_csv('./data/forget_data.csv')
df_all = df_all.drop_duplicates(subset=ID_COL, keep='first').reset_index(drop=True)

forget_ids = set(forget_df[ID_COL])
retain_df = df_all[~df_all[ID_COL].isin(forget_ids)].reset_index(drop=True)

# validation is carved out of the retain set only, never from Df
rng = np.random.default_rng(RANDOM_SEED)
perm = rng.permutation(len(retain_df))
n_val = int(0.10 * len(retain_df))
val_df = retain_df.iloc[perm[:n_val]].reset_index(drop=True)
train_df = retain_df.iloc[perm[n_val:]].reset_index(drop=True)

assert set(val_df[ID_COL]).isdisjoint(forget_ids)

print(f"train {len(train_df)} | val {len(val_df)} | forget {len(forget_df)}")

train 108629 | val 12069 | forget 9085
